In [1]:
import math
import os
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

ROOT = os.environ.get("AGL_ROOT", ".")


def P(*p):
    return os.path.join(ROOT, *p)


OUT_DIR = P("Code Outputs", "GNN Outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# config: identical to 18 and 19.ipynb
START, END = "1995-06-01", "2025-12-01"
TEST_MONTHS = 60
HORIZONS = [1, 3, 6, 12]
WINDOW = 12
H_HID, G_HID = 16, 24
EPOCHS, PATIENCE, LR, WD, DROP = 400, 30, 5e-3, 1e-4, 0.2
ENSEMBLE = 3                      # FC9 averages 3 networks
N_REP = int(os.environ.get("N_REP", 10))

SMOKE = os.environ.get("GNN_SMOKE") == "1"
if SMOKE:
    EPOCHS, PATIENCE, N_REP = 40, 40, 2
    print("*** SMOKE MODE - results are meaningless, code path test only ***")
SUF = "_SMOKE" if SMOKE else ""

# replicate r uses seeds (3r, 3r+1, 3r+2); replicate 0 is FC9's (0, 1, 2)
SEED_SETS = [[ENSEMBLE * r + k for k in range(ENSEMBLE)] for r in range(N_REP)]


def rmse(p, a):
    p, a = np.asarray(p, float), np.asarray(a, float)
    return np.sqrt(np.nanmean((p - a) ** 2))


def mae(p, a):
    p, a = np.asarray(p, float), np.asarray(a, float)
    return np.nanmean(np.abs(p - a))


# data
lev = pd.read_excel(P("Code Outputs", "Gap Interpolation Outputs",
                      "Unified_Interpolated_Levels.xlsx"))
lev["Date"] = pd.to_datetime(lev["Date"])
L = (lev.pivot(index="Date", columns="Reservoir", values="Level_m")
        .sort_index().asfreq("MS").loc[START:END])
LAKES = list(L.columns)
N = len(LAKES)
Lv = L.values
dL = np.diff(Lv, axis=0)
dates = L.index[1:]
T = len(dL)
split = T - TEST_MONTHS

mu, sd = dL[:split].mean(0), dL[:split].std(0) + 1e-8
dstd = ((dL - mu) / sd).astype(np.float32)
month = dates.month.values
sinm, cosm = np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12)
feats = np.stack([dstd,
                  np.repeat(sinm[:, None], N, 1),
                  np.repeat(cosm[:, None], N, 1)], axis=-1).astype(np.float32)
F_IN = feats.shape[-1]


def make_windows(lo, hi):
    X, Y = [], []
    for i in range(max(WINDOW, lo), hi):
        X.append(feats[i - WINDOW:i])
        Y.append(dstd[i])
    return np.array(X), np.array(Y)


val_start = int(split * 0.85)
Xtr, Ytr = make_windows(WINDOW, val_start)
Xva, Yva = make_windows(val_start, split)

# graphs
CORR = (pd.read_csv(P("Code Outputs", "EDA Outputs",
                      "EDA_06_corr_residual.csv"), index_col=0)
        .reindex(index=LAKES, columns=LAKES).values)
CCM = (pd.read_csv(P("Code Outputs", "CCM Outputs",
                     "CCM_09_adjacency_top3_diff_train.csv"), index_col=0)
       .reindex(index=LAKES, columns=LAKES).values.astype(float))
np.fill_diagonal(CCM, 0.0)


def sym_norm(A):
    A = np.asarray(A, float) + np.eye(N)
    d = np.abs(A).sum(1)
    Di = np.diag(1.0 / np.sqrt(d))
    return torch.tensor(Di @ A @ Di, dtype=torch.float32)


def row_norm(A):
    A = np.asarray(A, float) + np.eye(N)
    d = np.abs(A).sum(1, keepdims=True)
    return torch.tensor(A / d, dtype=torch.float32)


# the as-built graph, double self-loop and all
_abs_asbuilt = np.abs(CORR).copy()
_abs_asbuilt[_abs_asbuilt < 0.2] = 0.0        

CCM_SYM = np.maximum(CCM, CCM.T)              

GRAPHS = {
    "identity":           torch.eye(N, dtype=torch.float32),
    "ccm_symmetric":      sym_norm(CCM_SYM),
    "ccm_symmetric_row":  row_norm(CCM_SYM),   
    "ccm_directed":       row_norm(CCM),
    "corr_abs_asbuilt":   sym_norm(_abs_asbuilt),
}


# model
class GCN(nn.Module):
    def __init__(s, fin, fout):
        super().__init__()
        s.lin = nn.Linear(fin, fout)

    def forward(s, x, An):
        x = s.lin(x)
        x = torch.einsum("ij,bjf->bif", An, x)
        return torch.relu(x)


class STGNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.g1 = GCN(F_IN, H_HID)
        s.g2 = GCN(H_HID, H_HID)
        s.gru = nn.GRU(H_HID, G_HID, batch_first=True)
        s.drop = nn.Dropout(DROP)
        s.out = nn.Linear(G_HID, 1)

    def forward(s, x, An):
        B, Lw, n, f = x.shape
        h = x.reshape(B * Lw, n, f)
        h = s.g2(s.g1(h, An), An)
        h = h.reshape(B, Lw, n, -1).permute(0, 2, 1, 3).reshape(B * n, Lw, -1)
        _, hn = s.gru(h)
        o = s.out(s.drop(hn[-1]))
        return o.reshape(B, n)


def train_one(seed, An):
    torch.manual_seed(seed)
    np.random.seed(seed)
    m = STGNN()
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.MSELoss()
    xtr = torch.tensor(Xtr, dtype=torch.float32)
    ytr = torch.tensor(Ytr, dtype=torch.float32)
    xva = torch.tensor(Xva, dtype=torch.float32)
    yva = torch.tensor(Yva, dtype=torch.float32)
    best, best_state, wait = np.inf, None, 0
    for ep in range(EPOCHS):
        m.train()
        opt.zero_grad()
        loss = lossf(m(xtr, An), ytr)
        loss.backward()
        opt.step()
        m.eval()
        with torch.no_grad():
            vl = lossf(m(xva, An), yva).item()
        if vl < best - 1e-5:
            best, wait = vl, 0
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    m.load_state_dict(best_state)
    m.eval()
    return m, best


def evaluate(An, seeds):

    trained = [train_one(s, An) for s in seeds]
    models = [t[0] for t in trained]
    val_mses = [t[1] for t in trained]

    def predict_diff(w):
        x = torch.tensor(w[None], dtype=torch.float32)
        with torch.no_grad():
            return np.mean([mm(x, An).numpy()[0] for mm in models], axis=0)

    preds = {}
    for o in range(split, T):
        win = feats[o - WINDOW:o].copy()
        base = Lv[o]
        cum = 0.0
        Hh = min(max(HORIZONS), T - o)
        for k in range(Hh):
            pstd = predict_diff(win)
            cum = cum + (pstd * sd + mu)
            h = k + 1
            if h in HORIZONS:
                for j, lk in enumerate(LAKES):
                    preds[(lk, o + k, h)] = base[j] + cum[j]
            nxt = o + k
            mth = dates[nxt].month if nxt < T else ((dates[-1].month % 12) + 1)
            newf = np.stack([pstd,
                             np.full(N, np.sin(2 * np.pi * mth / 12), np.float32),
                             np.full(N, np.cos(2 * np.pi * mth / 12), np.float32)],
                            axis=-1)
            win = np.concatenate([win[1:], newf[None]], axis=0)

    rows = []
    for j, lk in enumerate(LAKES):
        for h in HORIZONS:
            Pv, Av = [], []
            for o in range(split, T):
                key = (lk, o + h - 1, h)
                if key in preds and (o + h) < len(Lv):
                    Pv.append(preds[key])
                    Av.append(Lv[o + h][j])
            rows.append({"Lake": lk, "Horizon_m": h,
                         "RMSE_m": round(rmse(Pv, Av), 4),
                         "MAE_m": round(mae(Pv, Av), 4),
                         "n_scored": len(Pv)})
    return pd.DataFrame(rows), val_mses


# SARIMA denominator, same source as FC9
FC6 = pd.read_csv(os.path.join(OUT_DIR, "FC6_gnn_metrics.csv"))
SAR = FC6[FC6.Model == "SARIMA"].set_index(["Lake", "Horizon_m"])["RMSE_m"]


def add_skill(df):
    df["skill_vs_SARIMA_%"] = df.apply(
        lambda r: round(100 * (SAR[(r.Lake, r.Horizon_m)] - r.RMSE_m)
                        / SAR[(r.Lake, r.Horizon_m)], 1), axis=1)
    return df


print("=" * 74)
print("GNN SEED VARIANCE  (FC10)")
print("=" * 74)
print(f"\n  graphs     : {', '.join(GRAPHS)}")
print(f"  replicates : {N_REP}  (ensemble of {ENSEMBLE} per replicate)")
print(f"  seed sets  : {SEED_SETS[0]} ... {SEED_SETS[-1]}")
print(f"  networks to train: {len(GRAPHS) * N_REP * ENSEMBLE}")
print(f"  estimated runtime: {len(GRAPHS)*N_REP*ENSEMBLE*13/60:.0f}-"
      f"{len(GRAPHS)*N_REP*ENSEMBLE*21/60:.0f} minutes\n")

rows = []
for gname, An in GRAPHS.items():
    line = []
    for r, seeds in enumerate(SEED_SETS):
        met, vmse = evaluate(An, seeds)
        met = add_skill(met)
        met.insert(0, "replicate", r)
        met.insert(0, "Graph", gname)
        met["seeds"] = ",".join(map(str, seeds))
        met["val_mse_mean"] = round(float(np.mean(vmse)), 5)
        rows.append(met)
        line.append(met[met.Horizon_m == 1]["skill_vs_SARIMA_%"].mean())
        print(f"    {gname:18s} rep {r:2d} seeds {str(seeds):12s} "
              f"h1={line[-1]:7.2f}")
    print(f"    {gname:18s} --> h1 mean {np.mean(line):7.2f}  "
          f"sd {np.std(line, ddof=1):5.2f}  "
          f"[{np.min(line):6.2f}, {np.max(line):6.2f}]\n")

VAR_ = pd.concat(rows, ignore_index=True)
VAR_.to_csv(os.path.join(OUT_DIR, f"FC10_gnn_seed_variance{SUF}.csv"), index=False)

# REGRESSION TEST
base = VAR_[(VAR_.Graph == "corr_abs_asbuilt") & (VAR_.replicate == 0)][
    ["Lake", "Horizon_m", "RMSE_m"]]
ref = FC6[FC6.Model == "GNN"][["Lake", "Horizon_m", "RMSE_m"]]
j = ref.merge(base, on=["Lake", "Horizon_m"], suffixes=("_fc6", "_new"))
assert len(j) == 28, f"expected 28 rows, got {len(j)}"
bad = int((j.RMSE_m_fc6 != j.RMSE_m_new).sum())
print(f"  regression test - replicate 0 vs FC6 GNN rows: {28 - bad}/28 identical")
if not SMOKE:
    assert bad == 0, ("replicate 0 does not reproduce FC6/FC9 - this harness "
                      "differs from the one that produced the published numbers")

# per-replicate mean skill
REP = (VAR_.groupby(["Graph", "replicate"] + ["Horizon_m"])["skill_vs_SARIMA_%"]
       .mean().reset_index())
WIDE = REP.pivot_table(index=["Graph", "replicate"], columns="Horizon_m",
                       values="skill_vs_SARIMA_%")
WIDE.columns = [f"h{c}" for c in WIDE.columns]
WIDE = WIDE.reset_index()

SUMM = (WIDE.groupby("Graph")[[f"h{h}" for h in HORIZONS]]
        .agg(["mean", "std", "min", "max"]).round(2))
SUMM.to_csv(os.path.join(OUT_DIR, f"FC10_seed_summary{SUF}.csv"))

print("\n" + "=" * 74)
print(f"MEAN skill vs SARIMA (%) over {N_REP} replicates, with seed spread")
print("=" * 74)
for h in HORIZONS:
    c = f"h{h}"
    print(f"\n  h={h}")
    print(f"    {'graph':20s} {'mean':>8s} {'sd':>7s} {'min':>8s} {'max':>8s} "
          f"{'FC9':>8s}")
    for g in GRAPHS:
        s = WIDE[WIDE.Graph == g][c]
        fc9 = WIDE[(WIDE.Graph == g) & (WIDE.replicate == 0)][c].iloc[0]
        print(f"    {g:20s} {s.mean():8.2f} {s.std(ddof=1):7.2f} "
              f"{s.min():8.2f} {s.max():8.2f} {fc9:8.2f}")

# PAIRED comparisons vs identity

try:
    from scipy import stats
    HAVE_SCIPY = True
except Exception:
    HAVE_SCIPY = False
    print("\n  [note] scipy unavailable - using a normal approximation for p")

pair_rows = []
print("\n" + "=" * 74)
print("PAIRED DIFFERENCES: identity minus graph  (positive = no graph is BETTER)")
print("=" * 74)
for h in HORIZONS:
    c = f"h{h}"
    ident = WIDE[WIDE.Graph == "identity"].set_index("replicate")[c]
    print(f"\n  h={h}")
    print(f"    {'graph':20s} {'mean diff':>10s} {'sd':>7s} {'95% CI':>18s} "
          f"{'t':>7s} {'p':>9s}")
    for g in GRAPHS:
        if g == "identity":
            continue
        other = WIDE[WIDE.Graph == g].set_index("replicate")[c]
        d = (ident - other).dropna()
        n = len(d)
        m_, s_ = d.mean(), d.std(ddof=1)
        se = s_ / np.sqrt(n) if n > 1 else np.nan
        t = m_ / se if se and se > 0 else np.nan
        if HAVE_SCIPY and n > 1:
            crit = stats.t.ppf(0.975, n - 1)
            p = 2 * (1 - stats.t.cdf(abs(t), n - 1))
        else:
            crit = 1.96
            p = (2 * (1 - 0.5 * (1 + math.erf(abs(t) / math.sqrt(2))))
                 if t == t else np.nan)
        lo, hi = m_ - crit * se, m_ + crit * se
        pair_rows.append({"Horizon_m": h, "graph": g, "n_replicates": n,
                          "mean_diff": round(m_, 3), "sd_diff": round(s_, 3),
                          "ci_lo": round(lo, 3), "ci_hi": round(hi, 3),
                          "t": round(t, 3), "p_value": round(p, 5)})
        print(f"    {g:20s} {m_:10.2f} {s_:7.2f} "
              f"[{lo:7.2f},{hi:7.2f}] {t:7.2f} {p:9.5f}")


print("\n" + "-" * 74)
print("CONFOUND CHECK (both use the paired replicates)")
print("-" * 74)
for h in HORIZONS:
    c = f"h{h}"
    sym = WIDE[WIDE.Graph == "ccm_symmetric"].set_index("replicate")[c]
    symr = WIDE[WIDE.Graph == "ccm_symmetric_row"].set_index("replicate")[c]
    dirg = WIDE[WIDE.Graph == "ccm_directed"].set_index("replicate")[c]
    dn = (symr - dirg).dropna()          # SAME normaliser, direction removed
    nn_ = (symr - sym).dropna()          # SAME graph, normaliser changed
    pair_rows.append({"Horizon_m": h, "graph": "DIRECTION_effect_rownorm",
                      "n_replicates": len(dn), "mean_diff": round(dn.mean(), 3),
                      "sd_diff": round(dn.std(ddof=1), 3), "ci_lo": np.nan,
                      "ci_hi": np.nan, "t": np.nan, "p_value": np.nan})
    pair_rows.append({"Horizon_m": h, "graph": "NORMALISATION_effect_ccmsym",
                      "n_replicates": len(nn_), "mean_diff": round(nn_.mean(), 3),
                      "sd_diff": round(nn_.std(ddof=1), 3), "ci_lo": np.nan,
                      "ci_hi": np.nan, "t": np.nan, "p_value": np.nan})
    print(f"  h={h:<3d} direction only (sym_row - directed) {dn.mean():7.2f} "
          f"+/- {dn.std(ddof=1):5.2f}   "
          f"normalisation only (sym_row - sym) {nn_.mean():7.2f} "
          f"+/- {nn_.std(ddof=1):5.2f}")

pd.DataFrame(pair_rows).to_csv(
    os.path.join(OUT_DIR, f"FC10_seed_paired{SUF}.csv"), index=False)

# figure 
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(15, 4.2), sharey=False)
for ax, h in zip(axes, HORIZONS):
    c = f"h{h}"
    order = list(GRAPHS)
    data = [WIDE[WIDE.Graph == g][c].values for g in order]
    ax.boxplot(data, widths=.55, showfliers=False)
    for i, g in enumerate(order):
        v = WIDE[WIDE.Graph == g][c].values
        ax.plot(np.full(len(v), i + 1) + np.random.uniform(-.08, .08, len(v)),
                v, "o", ms=3.5, alpha=.7,
                color="#C1440E" if g == "identity" else "#555555")
        fc9 = WIDE[(WIDE.Graph == g) & (WIDE.replicate == 0)][c].iloc[0]
        ax.plot([i + 1], [fc9], "*", ms=11, color="#1F5F8B", zorder=5)
    ax.axhline(0, color="k", lw=1)
    ax.set_xticklabels(order, rotation=40, ha="right", fontsize=7.5)
    ax.set_title(f"h={h}")
    ax.grid(alpha=.3)
axes[0].set_ylabel("mean skill vs SARIMA (%)")
plt.suptitle(f"GNN adjacency: seed variability over {N_REP} independent "
             f"{ENSEMBLE}-seed replicates   (blue star = the FC9 replicate)",
             fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"FC10_seed_variance{SUF}.png"), dpi=200)
plt.close()

print("\nWritten to", OUT_DIR)
for f in (f"FC10_gnn_seed_variance{SUF}.csv", f"FC10_seed_summary{SUF}.csv",
          f"FC10_seed_paired{SUF}.csv", f"FC10_seed_variance{SUF}.png"):
    print("   ", f)
print("\nHOW TO READ THE RESULT")
print("  If the identity-minus-graph paired differences stay clearly positive")
print("  with CIs excluding zero, FC9's conclusion holds and now has an error")
print("  bar. If the CIs straddle zero, the FC9 ranking was seed noise and the")
print("  Results chapter must say so.")

GNN SEED VARIANCE  (FC10)

  graphs     : identity, ccm_symmetric, ccm_symmetric_row, ccm_directed, corr_abs_asbuilt
  replicates : 10  (ensemble of 3 per replicate)
  seed sets  : [0, 1, 2] ... [27, 28, 29]
  networks to train: 150
  estimated runtime: 32-52 minutes

    identity           rep  0 seeds [0, 1, 2]    h1=  -3.54
    identity           rep  1 seeds [3, 4, 5]    h1=  -8.89
    identity           rep  2 seeds [6, 7, 8]    h1= -12.14
    identity           rep  3 seeds [9, 10, 11]  h1= -13.80
    identity           rep  4 seeds [12, 13, 14] h1=  -6.47
    identity           rep  5 seeds [15, 16, 17] h1= -12.13
    identity           rep  6 seeds [18, 19, 20] h1= -13.11
    identity           rep  7 seeds [21, 22, 23] h1=  -6.86
    identity           rep  8 seeds [24, 25, 26] h1= -11.00
    identity           rep  9 seeds [27, 28, 29] h1=  -4.77
    identity           --> h1 mean   -9.27  sd  3.68  [-13.80,  -3.54]

    ccm_symmetric      rep  0 seeds [0, 1, 2]    h1= -11.19